# Plant Disease Dataset - Data Cleaning & Analysis Dashboard

This notebook provides a complete, step-by-step pipeline to clean and validate the plant disease image dataset:


In [16]:
# Step 1: Setup & Configuration
import os
import sys
import hashlib
from datetime import datetime
from pathlib import Path
from PIL import Image
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from tqdm import tqdm

# Define global configuration constants
DATASET_PATH = Path(r"D:\Projects\plantdisease\dataset")
EXCEL_FILE = Path(r"D:\Projects\plantdisease\image_count_report.xlsx")
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

print("✓ Environment set up successfully!")
print(f"Dataset Path:        {DATASET_PATH}")
print(f"Excel Report Path:   {EXCEL_FILE}")
print(f"Supported Formats:   {', '.join(sorted(SUPPORTED_EXTENSIONS))}")

✓ Environment set up successfully!
Dataset Path:        D:\Projects\plantdisease\dataset
Excel Report Path:   D:\Projects\plantdisease\image_count_report.xlsx
Supported Formats:   .bmp, .jpeg, .jpg, .png, .tiff, .webp


# Step 2: Dataset Class Distribution Scan
Scan all classes under the dataset folder, count the number of images in each class, and write the counts to the `Image_Count` sheet of the Excel report.

In [17]:
# Step 2: Class Count Scan & Excel Output
print("Scanning dataset structure...")
print("=" * 80)

dataset_info = []

# Loop through crop folders
for crop_dir in sorted(DATASET_PATH.iterdir()):
    if not crop_dir.is_dir():
        continue
    
    # Loop through class folders within the crop
    for class_dir in sorted(crop_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        
        # Count images in this class
        image_count = len([
            f for f in class_dir.iterdir() 
            if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
        ])
        
        dataset_info.append({
            "Crop": crop_dir.name,
            "Class": class_dir.name,
            "Image Count": image_count
        })

df_structure = pd.DataFrame(dataset_info)
print(f"✓ Found {len(df_structure)} crop-class combinations.")
print("=" * 80)

# Write to Excel (preserving other sheets if file exists)
print("\nWriting class counts to Excel...")
if EXCEL_FILE.exists():
    wb = load_workbook(EXCEL_FILE)
    if "Image_Count" in wb.sheetnames:
        del wb["Image_Count"]
    ws = wb.create_sheet("Image_Count")
else:
    from openpyxl import Workbook
    wb = Workbook()
    wb.remove(wb.active)
    ws = wb.create_sheet("Image_Count", 0)

# Write headers and data
headers = ["Crop", "Class", "Image Count"]
ws.append(headers)
for row in df_structure.itertuples(index=False, name=None):
    ws.append(row)

# Apply formatting
header_fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
header_font = Font(bold=True)

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Adjust column widths
for column in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

ws.freeze_panes = "A2"
if ws.max_row > 1:
    ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"

try:
    wb.save(EXCEL_FILE)
    print(f"✓ Successfully saved 'Image_Count' to: {EXCEL_FILE}")
except PermissionError:
    import time
    backup_path = EXCEL_FILE.parent / f"{EXCEL_FILE.stem}_backup_{int(time.time())}{EXCEL_FILE.suffix}"
    wb.save(backup_path)
    print(f"⚠ Excel file locked. Saved to backup: {backup_path}")

Scanning dataset structure...
✓ Found 48 crop-class combinations.

Writing class counts to Excel...
✓ Successfully saved 'Image_Count' to: D:\Projects\plantdisease\image_count_report.xlsx


# Step 3: Dataset Summary Statistics
Calculate and display high-level statistics of the dataset grouped by Crop.

In [18]:
# Step 3: Statistics Summary Display
print("=" * 80)
print("DATASET STRUCTURAL SUMMARY")
print("=" * 80)

total_images = df_structure["Image Count"].sum()
total_crops = df_structure["Crop"].nunique()
total_classes = len(df_structure)

print(f"Total Crops:   {total_crops}")
print(f"Total Classes: {total_classes}")
print(f"Total Images:  {total_images:,}")
print("-" * 50)

# Group and display by crop
crop_summary = df_structure.groupby("Crop").agg({
    "Class": "count",
    "Image Count": "sum"
}).rename(columns={"Class": "Classes", "Image Count": "Images"})

print(f"{'Crop':<20} {'Classes':<10} {'Images':<15}")
print("-" * 50)
for crop, row in crop_summary.iterrows():
    print(f"{crop:<20} {row['Classes']:<10} {row['Images']:<15,}")
print("=" * 80)

DATASET STRUCTURAL SUMMARY
Total Crops:   8
Total Classes: 48
Total Images:  43,601
--------------------------------------------------
Crop                 Classes    Images         
--------------------------------------------------
banana               4          4,675          
corn                 4          4,188          
cotton               4          1,710          
mango                8          4,000          
potato               7          3,500          
rice                 6          3,829          
sugarcane            5          2,521          
tomato               10         19,178         


# Step 4: Corrupted Image Validation
Validate every image in the dataset to verify that:
1. It is not empty (size > 0 bytes).
2. It can be opened with PIL.
3. It has valid headers.
4. Its pixels load fully (identifies truncated or corrupted image content).

Any invalid image is logged to Excel in detail.

In [19]:
# Step 4: Validation Logic & Execution
def check_image_corruption(file_path: Path) -> tuple:
    """Validate image file. Returns (is_valid, error_type, error_message)."""
    try:
        if not file_path.exists():
            return False, "File Not Found", "File does not exist"
        
        file_size = file_path.stat().st_size
        if file_size == 0:
            return False, "Zero Byte File", "File is empty"
            
        try:
            with Image.open(file_path) as img:
                if img.format is None:
                    return False, "Invalid Header", "Cannot identify format"
                img.verify()
        except IOError as e:
            err_msg = str(e).lower()
            if "truncated" in err_msg:
                return False, "Truncated Image", str(e)
            return False, "Invalid Header", str(e)
            
        # Verify complete data loading
        try:
            with Image.open(file_path) as img:
                img.load()
                if img.size == (0, 0):
                    return False, "Invalid Header", "Zero dimensions"
        except Exception as e:
            if "truncated" in str(e).lower():
                return False, "Truncated Image", str(e)
            return False, "Verification Failed", str(e)
            
        return True, "Valid", "Image is valid"
    except Exception as e:
        return False, "Unknown Error", str(e)

# Gather and check all images
all_image_files = sorted([
    f for f in DATASET_PATH.rglob("*") 
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
])

print(f"Validating {len(all_image_files):,} images...")
corrupted_images = []
valid_count = 0
scan_start = datetime.now()

for file_path in tqdm(all_image_files, desc="Validating", unit=" images"):
    is_valid, err_type, err_msg = check_image_corruption(file_path)
    if is_valid:
        valid_count += 1
    else:
        corrupted_images.append({
            "File Path": str(file_path),
            "File Name": file_path.name,
            "File Extension": file_path.suffix.lower(),
            "File Size (KB)": round(file_path.stat().st_size / 1024, 2),
            "Error Type": err_type,
            "Error Message": err_msg
        })

scan_duration = (datetime.now() - scan_start).total_seconds()
print("\n" + "=" * 80)
print(f"✓ Validation Complete in {scan_duration:.2f} seconds!")
print(f"  Valid Images:     {valid_count:,}")
print(f"  Corrupted Images: {len(corrupted_images):,}")
print("=" * 80)

Validating 43,601 images...


Validating: 100%|██████████| 43601/43601 [00:54<00:00, 806.51 images/s] 


✓ Validation Complete in 54.06 seconds!
  Valid Images:     43,601
  Corrupted Images: 0


In [20]:
# Step 4: Corrupted Reports Excel Output
print("Writing corruption reports to Excel...")

if EXCEL_FILE.exists():
    wb = load_workbook(EXCEL_FILE)
    if "Corrupted_Images" in wb.sheetnames:
        del wb["Corrupted_Images"]
    if "Corrupted_Image_Summary" in wb.sheetnames:
        del wb["Corrupted_Image_Summary"]
else:
    wb = Workbook()
    wb.remove(wb.active)

# 1. Detailed sheet
ws_details = wb.create_sheet("Corrupted_Images")
headers_details = ["File Path", "File Name", "File Extension", "File Size (KB)", "Error Type", "Error Message"]
ws_details.append(headers_details)

if corrupted_images:
    for item in corrupted_images:
        ws_details.append([item["File Path"], item["File Name"], item["File Extension"], item["File Size (KB)"], item["Error Type"], item["Error Message"]])
else:
    ws_details.append(["No corrupted images found!", "", "", "", "", ""])

# Style details
ws_details.freeze_panes = "A2"
for cell in ws_details[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
for column in ws_details.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws_details.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)
if ws_details.max_row > 1:
    ws_details.auto_filter.ref = f"A1:{get_column_letter(ws_details.max_column)}{ws_details.max_row}"

# 2. Summary sheet
ws_summary = wb.create_sheet("Corrupted_Image_Summary")
ws_summary.append(["Metric", "Value"])
corruption_rate = (len(corrupted_images) / len(all_image_files) * 100) if all_image_files else 0

summary_rows = [
    ["Total Images Scanned", len(all_image_files)],
    ["Valid Images", valid_count],
    ["Corrupted Images", len(corrupted_images)],
    ["Corruption Percentage (%)", round(corruption_rate, 2)],
    ["Scan Duration (seconds)", round(scan_duration, 2)],
    ["Scan Date & Time", datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
]
for row in summary_rows:
    ws_summary.append(row)

# Style summary
for cell in ws_summary[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
for cell in ws_summary["A"]:
    cell.font = Font(bold=True)
for column in ws_summary.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws_summary.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

try:
    wb.save(EXCEL_FILE)
    print(f"✓ Saved corruption reports to: {EXCEL_FILE}")
except PermissionError:
    import time
    backup_path = EXCEL_FILE.parent / f"{EXCEL_FILE.stem}_corrupt_backup_{int(time.time())}{EXCEL_FILE.suffix}"
    wb.save(backup_path)
    print(f"⚠ Excel file locked. Saved to backup: {backup_path}")

Writing corruption reports to Excel...
✓ Saved corruption reports to: D:\Projects\plantdisease\image_count_report.xlsx


# Step 5: Exact Duplicate Detection and Reporting
Identify identical files by calculating MD5 hashes. Log duplicate groups to `duplicate_groups.log` and write sheets `Exact_Duplicate_Report` and `Exact_Duplicate_Summary` in the Excel report.

In [21]:
# Step 5: Hashing and Duplicate Grouping
def calculate_md5(file_path: Path) -> str:
    """Compute MD5 of a file in 64KB blocks."""
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"Error hashing {file_path.name}: {e}")
        return ""

print("Hashing all images to detect duplicates...")
image_hashes = []
hash_start = datetime.now()

for file_path in tqdm(all_image_files, desc="Hashing", unit=" images"):
    md5_val = calculate_md5(file_path)
    if md5_val:
        # Extract crop and class from path
        try:
            rel_path = file_path.relative_to(DATASET_PATH)
            crop = rel_path.parts[0] if len(rel_path.parts) > 1 else "unknown"
            cls = rel_path.parts[1] if len(rel_path.parts) > 2 else "unknown"
        except Exception:
            crop, cls = "unknown", "unknown"
            
        image_hashes.append({
            "File Path": str(file_path),
            "File Name": file_path.name,
            "Crop": crop,
            "Class": cls,
            "File Size (KB)": round(file_path.stat().st_size / 1024, 2),
            "MD5": md5_val
        })

hash_duration = (datetime.now() - hash_start).total_seconds()
df_hashes = pd.DataFrame(image_hashes)

# Identify duplicate groups
hash_counts = df_hashes["MD5"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

df_duplicates = df_hashes[df_hashes["MD5"].isin(duplicate_hashes)].copy().sort_values("MD5")
md5_to_group_id = {md5: f"Group_{i+1:04d}" for i, md5 in enumerate(duplicate_hashes)}
df_duplicates["Duplicate Group ID"] = df_duplicates["MD5"].map(md5_to_group_id)

cols_order = ["Duplicate Group ID", "Crop", "Class", "File Name", "File Size (KB)", "MD5", "File Path"]
df_duplicate_report = df_duplicates[cols_order].copy()

total_groups = len(duplicate_hashes)
total_duplicate_files = len(df_duplicates)
redundant_files = total_duplicate_files - total_groups

print("\n" + "=" * 80)
print(f"✓ Hashing and Detection Complete in {hash_duration:.2f} seconds!")
print(f"  Duplicate Groups Found: {total_groups:,}")
print(f"  Total Duplicate Files:  {total_duplicate_files:,}")
print(f"  Redundant Files (Removable): {redundant_files:,}")
print("=" * 80)

Hashing all images to detect duplicates...


Hashing: 100%|██████████| 43601/43601 [00:09<00:00, 4754.22 images/s]



✓ Hashing and Detection Complete in 9.17 seconds!
  Duplicate Groups Found: 751
  Total Duplicate Files:  1,555
  Redundant Files (Removable): 804


In [22]:
# Step 5: Logging duplicate groups for review
DUPLICATE_LOG_FILE = Path(r"D:\Projects\plantdisease\duplicate_groups.log")
print(f"Logging duplicate groups to: {DUPLICATE_LOG_FILE}")

with open(DUPLICATE_LOG_FILE, "w", encoding="utf-8") as f:
    f.write("================================================================================\n")
    f.write(f"EXACT DUPLICATE GROUPS LOG - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("================================================================================\n")
    f.write(f"Total Duplicate Groups: {total_groups:,}\n")
    f.write(f"Total Files in Groups:  {total_duplicate_files:,}\n")
    f.write(f"Redundant Files:        {redundant_files:,}\n")
    f.write("================================================================================\n\n")
    
    for group_id in sorted(md5_to_group_id.values()):
        group_df = df_duplicate_report[df_duplicate_report["Duplicate Group ID"] == group_id]
        md5_hash = group_df.iloc[0]["MD5"]
        file_size = group_df.iloc[0]["File Size (KB)"]
        
        f.write(f"[{group_id}] MD5: {md5_hash} | Size: {file_size} KB | Files: {len(group_df)}\n")
        for idx, row in group_df.iterrows():
            f.write(f"  - [{row['Crop']} / {row['Class']}] {row['File Name']}\n")
            f.write(f"    Path: {row['File Path']}\n")
        f.write("-" * 80 + "\n")

print("✓ Log file created successfully.")

Logging duplicate groups to: D:\Projects\plantdisease\duplicate_groups.log
✓ Log file created successfully.


In [23]:
# Step 5: Excel Report & Summary Display
print("Writing duplicate reports to Excel...")

if EXCEL_FILE.exists():
    wb = load_workbook(EXCEL_FILE)
    if "Exact_Duplicate_Report" in wb.sheetnames:
        del wb["Exact_Duplicate_Report"]
    if "Exact_Duplicate_Summary" in wb.sheetnames:
        del wb["Exact_Duplicate_Summary"]
else:
    wb = Workbook()
    wb.remove(wb.active)

# 1. Report Sheet
ws_report = wb.create_sheet("Exact_Duplicate_Report")
if not df_duplicate_report.empty:
    headers_rep = list(df_duplicate_report.columns)
    ws_report.append(headers_rep)
    for row in df_duplicate_report.itertuples(index=False, name=None):
        ws_report.append(row)
else:
    ws_report.append(["Duplicate Group ID", "Crop", "Class", "File Name", "File Size (KB)", "MD5", "File Path"])
    ws_report.append(["No duplicate images found!", "", "", "", "", "", ""])

# Style Report
for cell in ws_report[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
for column in ws_report.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws_report.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)
ws_report.freeze_panes = "A2"
if ws_report.max_row > 1:
    ws_report.auto_filter.ref = f"A1:{get_column_letter(ws_report.max_column)}{ws_report.max_row}"

# 2. Summary Sheet
ws_summary = wb.create_sheet("Exact_Duplicate_Summary")
ws_summary.append(["Metric", "Value"])

saved_bytes = 0
for md5 in duplicate_hashes:
    group_df = df_hashes[df_hashes["MD5"] == md5]
    saved_bytes += (len(group_df) - 1) * group_df.iloc[0]["File Size (KB)"] * 1024
saved_mb = round(saved_bytes / (1024 * 1024), 2)

summary_rows = [
    ["Total Images Scanned", len(df_hashes)],
    ["Unique Images (MD5)", df_hashes["MD5"].nunique()],
    ["Duplicate Groups Found", total_groups],
    ["Total Files in Duplicate Groups", total_duplicate_files],
    ["Redundant Files (Removable)", redundant_files],
    ["Potential Space Savings (MB)", saved_mb],
    ["Scan Date & Time", datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    ["Scan Duration (seconds)", round(hash_duration, 2)],
    ["Dataset Path", str(DATASET_PATH)]
]
for row in summary_rows:
    ws_summary.append(row)

# Style Summary
for cell in ws_summary[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
for cell in ws_summary["A"]:
    cell.font = Font(bold=True)
for column in ws_summary.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws_summary.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

try:
    wb.save(EXCEL_FILE)
    print(f"✓ Saved duplicate reports to: {EXCEL_FILE}")
except PermissionError:
    import time
    backup_path = EXCEL_FILE.parent / f"{EXCEL_FILE.stem}_dup_backup_{int(time.time())}{EXCEL_FILE.suffix}"
    wb.save(backup_path)
    print(f"⚠ Excel file locked. Saved to backup: {backup_path}")

# Display Summary
print("\n")
print("╔" + "═" * 78 + "╗")
print("║" + " " * 22 + "EXACT DUPLICATE DETECTION REPORT" + " " * 24 + "║")
print("╚" + "═" * 78 + "╝")
print(f"  • Total Images Scanned:        {len(df_hashes):>12,}")
print(f"  • Unique Images (MD5):         {df_hashes['MD5'].nunique():>12,}")
print(f"  • Duplicate Groups:            {total_groups:>12,}")
print(f"  • Total Duplicate Files:       {total_duplicate_files:>12,}")
print(f"  • Redundant Files:             {redundant_files:>12,}")
print(f"  • Potential Space Savings:     {saved_mb:>9} MB")
print(f"  • Scan Duration:               {hash_duration:>12.2f} seconds")
print("╚" + "═" * 78 + "╝\n")

Writing duplicate reports to Excel...
✓ Saved duplicate reports to: D:\Projects\plantdisease\image_count_report.xlsx


╔══════════════════════════════════════════════════════════════════════════════╗
║                      EXACT DUPLICATE DETECTION REPORT                        ║
╚══════════════════════════════════════════════════════════════════════════════╝
  • Total Images Scanned:              43,601
  • Unique Images (MD5):               42,797
  • Duplicate Groups:                     751
  • Total Duplicate Files:              1,555
  • Redundant Files:                      804
  • Potential Space Savings:         78.99 MB
  • Scan Duration:                       9.17 seconds
╚══════════════════════════════════════════════════════════════════════════════╝



# Step 6: Safely Delete Redundant Duplicate Images
This step goes through all detected duplicate groups, keeps the first image (sorted alphabetically by path) as the original, and deletes all other duplicate files to clean up the dataset and reclaim space.

In [24]:
# Step 6: Safely Delete Redundant Duplicate Images
print("Safely deleting redundant duplicate images...")
print("=" * 80)

deleted_files_count = 0
deleted_bytes = 0
deleted_log = []

# Process each duplicate group
for group_id in tqdm(sorted(md5_to_group_id.values()), desc="Deleting duplicates"):
    # Get all files in this group, sorted alphabetically by file path
    group_df = df_duplicate_report[df_duplicate_report["Duplicate Group ID"] == group_id].sort_values("File Path")
    
    # Keep the first file, delete the rest
    keep_file = Path(group_df.iloc[0]["File Path"])
    files_to_delete = [Path(p) for p in group_df.iloc[1:]["File Path"]]
    
    for file_path in files_to_delete:
        if file_path.exists():
            try:
                file_size = file_path.stat().st_size
                file_path.unlink()  # Delete the file
                deleted_files_count += 1
                deleted_bytes += file_size
                deleted_log.append({
                    "Duplicate Group ID": group_id,
                    "Deleted File Name": file_path.name,
                    "Deleted File Path": str(file_path),
                    "File Size (KB)": round(file_size / 1024, 2),
                    "Kept Original Path": str(keep_file)
                })
            except Exception as e:
                print(f"Error deleting {file_path.name}: {e}")

deleted_mb = round(deleted_bytes / (1024 * 1024), 2)
print("\n" + "=" * 80)
print("✓ Deletion Process Complete!")
print(f"  Duplicate Groups Cleaned: {total_groups:,}")
print(f"  Redundant Files Deleted:  {deleted_files_count:,}")
print(f"  Total Reclaimed Space:    {deleted_mb:.2f} MB")
print("=" * 80)

Safely deleting redundant duplicate images...


Deleting duplicates: 100%|██████████| 751/751 [00:00<00:00, 1390.58it/s]


✓ Deletion Process Complete!
  Duplicate Groups Cleaned: 751
  Redundant Files Deleted:  804
  Total Reclaimed Space:    78.99 MB


# Step 7: Post-Cleaning Class Distribution Scan
Scan the dataset structure again after duplicate deletion, count the number of images in each class, and write the post-cleaning counts to a new worksheet `Image_Count_Post_Cleaning` in the Excel report.

In [26]:
# Step 7: Post-Cleaning Class Distribution Scan & Excel Output
print("Scanning dataset structure after duplicate deletion...")
print("=" * 80)

post_dataset_info = []

# Loop through crop folders
for crop_dir in sorted(DATASET_PATH.iterdir()):
    if not crop_dir.is_dir():
        continue
    
    # Loop through class folders within the crop
    for class_dir in sorted(crop_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        
        # Count images remaining in this class
        image_count = len([
            f for f in class_dir.iterdir() 
            if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
        ])
        
        post_dataset_info.append({
            "Crop": crop_dir.name,
            "Class": class_dir.name,
            "Image Count": image_count
        })

df_post_structure = pd.DataFrame(post_dataset_info)
print(f"✓ Found {len(df_post_structure)} crop-class combinations.")
print("=" * 80)

# Write to Excel (preserving other sheets)
print("\nWriting post-cleaning class counts to Excel...")
if EXCEL_FILE.exists():
    wb = load_workbook(EXCEL_FILE)
    if "Image_Count_Post_Cleaning" in wb.sheetnames:
        del wb["Image_Count_Post_Cleaning"]
    ws_post = wb.create_sheet("Image_Count_Post_Cleaning")
else:
    from openpyxl import Workbook
    wb = Workbook()
    wb.remove(wb.active)
    ws_post = wb.create_sheet("Image_Count_Post_Cleaning", 0)

# Write headers and data
headers = ["Crop", "Class", "Image Count"]
ws_post.append(headers)
for row in df_post_structure.itertuples(index=False, name=None):
    ws_post.append(row)

# Apply formatting
for cell in ws_post[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Adjust column widths
for column in ws_post.columns:
    max_len = max(len(str(cell.value or '')) for cell in column)
    ws_post.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)

ws_post.freeze_panes = "A2"
if ws_post.max_row > 1:
    ws_post.auto_filter.ref = f"A1:{get_column_letter(ws_post.max_column)}{ws_post.max_row}"

try:
    wb.save(EXCEL_FILE)
    print(f"✓ Successfully saved 'Image_Count_Post_Cleaning' to: {EXCEL_FILE}")
except PermissionError:
    import time
    backup_path = EXCEL_FILE.parent / f"{EXCEL_FILE.stem}_post_clean_backup_{int(time.time())}{EXCEL_FILE.suffix}"
    wb.save(backup_path)
    print(f"⚠ Excel file locked. Saved to backup: {backup_path}")

# Display summary post-cleaning stats
print("\n" + "=" * 80)
print(f"Post-Cleaning Total Images: {df_post_structure['Image Count'].sum():,}")
print(f"Total Images Deleted:       {df_structure['Image Count'].sum() - df_post_structure['Image Count'].sum():,}")
print("=" * 80)

Scanning dataset structure after duplicate deletion...
✓ Found 48 crop-class combinations.

Writing post-cleaning class counts to Excel...
✓ Successfully saved 'Image_Count_Post_Cleaning' to: D:\Projects\plantdisease\image_count_report.xlsx

Post-Cleaning Total Images: 42,797
Total Images Deleted:       804


# Step 8: Dataset Metadata Extraction & Class Distribution
Recursively scan all images in the dataset to extract file information, metadata, and folder-derived labels:
1. **Folder Parsing**: Extract the crop and disease names from the class folder naming convention (e.g., `Banana_Cordana` -> Crop: `Banana`, Disease: `Cordana`).
2. **Pillow Optimization**: Fetch image dimensions (width, height) efficiently by only reading headers (lazy loading) without loading full pixel data into memory.
3. **CSV Reports**: Export the compiled dataset metadata to `dataset_metadata.csv` and the summary class distribution to `class_distribution.csv`.

In [8]:
# Step 9: Dataset Metadata Extraction & Class Distribution
import logging
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Configure logging for metadata extraction
logger = logging.getLogger("DatasetMetadata")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

# Console log handler
c_handler = logging.StreamHandler()
c_handler.setFormatter(formatter)
logger.addHandler(c_handler)

# File log handler
log_file_path = Path(r"D:\Projects\plantdisease\dataset_metadata.log")
f_handler = logging.FileHandler(log_file_path, mode='w', encoding='utf-8')
f_handler.setFormatter(formatter)
logger.addHandler(f_handler)

# Configuration fallbacks
if 'DATASET_PATH' not in globals():
    DATASET_PATH = Path(r"D:\Projects\plantdisease\dataset")
if 'SUPPORTED_EXTENSIONS' not in globals():
    SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

METADATA_CSV = Path(r"D:\Projects\plantdisease\dataset_metadata.csv")
DISTRIBUTION_CSV = Path(r"D:\Projects\plantdisease\class_distribution.csv")

logger.info("Starting dataset metadata extraction...")
logger.info(f"Dataset Path: {DATASET_PATH}")

# Gather all image files recursively
image_files = sorted([
    f for f in DATASET_PATH.rglob("*") 
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
])

logger.info(f"Found {len(image_files):,} image files to process.")

metadata_records = []
failed_count = 0

# Process images with progress bar
for file_path in tqdm(image_files, desc="Extracting Metadata", unit=" images"):
    try:
        # Parse folder structure: DATASET_PATH / crop_folder / class_folder / image
        try:
            rel_parts = file_path.relative_to(DATASET_PATH).parts
            class_name = rel_parts[-2] if len(rel_parts) >= 2 else file_path.parent.name
        except Exception:
            class_name = file_path.parent.name
            
        parts = class_name.split("_", 1)
        crop = parts[0]
        disease = parts[1] if len(parts) > 1 else "Healthy"
        
        # Get file size in KB
        file_size_kb = round(file_path.stat().st_size / 1024, 2)
        
        # Get dimensions with Pillow (header-only lazy loading for speed)
        with Image.open(file_path) as img:
            width, height = img.size
            
        metadata_records.append({
            "image_path": str(file_path),
            "image_name": file_path.name,
            "crop": crop,
            "disease": disease,
            "class_name": class_name,
            "width": width,
            "height": height,
            "file_size_kb": file_size_kb
        })
        
    except Exception as e:
        failed_count += 1
        logger.error(f"Failed to process image {file_path}: {e}")

logger.info(f"Metadata extraction complete. Failed: {failed_count}")

if metadata_records:
    # Convert to DataFrame
    df = pd.DataFrame(metadata_records)
    
    # Save full metadata CSV
    df.to_csv(METADATA_CSV, index=False, encoding='utf-8')
    logger.info(f"Saved dataset metadata to {METADATA_CSV}")
    
    # Save class distribution CSV
    class_counts = df['class_name'].value_counts().reset_index()
    class_counts.columns = ['class_name', 'image_count']
    class_counts = class_counts.sort_values('class_name')
    class_counts.to_csv(DISTRIBUTION_CSV, index=False, encoding='utf-8')
    logger.info(f"Saved class distribution to {DISTRIBUTION_CSV}")
    
    # Print neat summary statistics
    total_images = len(df)
    total_crops = df['crop'].nunique()
    total_classes = df['class_name'].nunique()
    
    print("\n" + "=" * 80)
    print("DATASET METADATA SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total Images:      {total_images:,}")
    print(f"Total Crops:       {total_crops}")
    print(f"Total Classes:     {total_classes}")
    
    print("\nImages Per Crop:")
    print("-" * 30)
    crop_counts = df['crop'].value_counts().sort_index()
    for crop_name, count in crop_counts.items():
        print(f"  {crop_name:<20} {count:>8,}")
        
    print("\nImages Per Class (Top 10):")
    print("-" * 35)
    top_classes = df['class_name'].value_counts().head(10)
    for class_name, count in top_classes.items():
        print(f"  {class_name:<25} {count:>8,}")
    if total_classes > 10:
        print(f"  ... and {total_classes - 10} more classes (see class_distribution.csv for full list)")
    print("=" * 80)
else:
    logger.warning("No metadata records were extracted. No CSV files created.")


2026-06-22 22:30:08,049 - INFO - Starting dataset metadata extraction...
2026-06-22 22:30:08,050 - INFO - Dataset Path: D:\Projects\plantdisease\dataset
2026-06-22 22:30:11,964 - INFO - Found 42,797 image files to process.
Extracting Metadata: 100%|██████████| 42797/42797 [00:15<00:00, 2685.62 images/s]
2026-06-22 22:30:27,905 - INFO - Metadata extraction complete. Failed: 0
2026-06-22 22:30:28,300 - INFO - Saved dataset metadata to D:\Projects\plantdisease\dataset_metadata.csv
2026-06-22 22:30:28,343 - INFO - Saved class distribution to D:\Projects\plantdisease\class_distribution.csv



DATASET METADATA SUMMARY STATISTICS
Total Images:      42,797
Total Crops:       8
Total Classes:     48

Images Per Crop:
------------------------------
  Banana                  4,334
  Corn                    4,186
  Cotton                  1,591
  Mango                   3,979
  Potato                  3,420
  Rice                    3,798
  Sugarcane               2,311
  Tomato                 19,178

Images Per Class (Top 10):
-----------------------------------
  Tomato_YellowLeafCurlVirus    5,357
  Banana_YellowBlackSigatoka    2,162
  Tomato_BacterialSpot         2,127
  Tomato_LateBlight            1,905
  Tomato_SeptoriaLeafSpot      1,780
  Tomato_SpiderMitesTwoSpottedSpiderMite    1,690
  Tomato_Healthy               1,603
  Tomato_TargetSpot            1,433
  Corn_CommonRust              1,306
  Corn_Healthy                 1,162
  ... and 38 more classes (see class_distribution.csv for full list)


# Step 9: Dataset Split — Train / Validation / Test

Split the cleaned dataset into **70% train**, **15% validation**, and **15% test** sets.

- Each class folder inside the dataset is treated as one class.
- Images are randomly shuffled with a fixed seed (42) for reproducibility.
- The split is performed **independently per class** to preserve class distribution.
- Images are **copied** (not moved) into `dataset_split/train|val|test/<class>/`.
- The original dataset is left completely untouched.

In [2]:
# Step 13.1 — Split Setup & Configuration
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ── Paths ──────────────────────────────────────────────────────────────────
DATASET_PATH         = Path(r"D:\Projects\plantdisease\dataset")
SPLIT_ROOT           = Path(r"D:\Projects\plantdisease\dataset_split")
EXCEL_FILE           = Path(r"D:\Projects\plantdisease\image_count_report.xlsx")
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

# ── Split ratios & seed ────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SEED        = 42

# Create output split directories
for split in ("train", "val", "test"):
    (SPLIT_ROOT / split).mkdir(parents=True, exist_ok=True)

print("✓ Split directories created:")
print(f"  {SPLIT_ROOT / 'train'}")
print(f"  {SPLIT_ROOT / 'val'}")
print(f"  {SPLIT_ROOT / 'test'}")

✓ Split directories created:
  D:\Projects\plantdisease\dataset_split\train
  D:\Projects\plantdisease\dataset_split\val
  D:\Projects\plantdisease\dataset_split\test


In [3]:
# Step 13.2 — Collect Classes & Perform the Split

# Collect all class directories (handles both flat and one-level-nested layouts)
class_dirs = []
for entry in sorted(DATASET_PATH.iterdir()):
    if not entry.is_dir():
        continue
    # Check if this folder contains images directly (flat layout)
    has_images = any(
        f.suffix.lower() in SUPPORTED_EXTENSIONS
        for f in entry.iterdir() if f.is_file()
    )
    if has_images:
        class_dirs.append(entry)          # flat:   dataset/ClassName/
    else:
        for sub in sorted(entry.iterdir()):  # nested: dataset/crop/ClassName/
            if sub.is_dir():
                class_dirs.append(sub)

print(f"Found {len(class_dirs)} classes.\n")

split_records = []  # for the Excel report

for class_dir in tqdm(class_dirs, desc="Splitting classes", unit=" class"):
    class_name = class_dir.name

    # Gather all valid images in this class
    images = sorted([
        f for f in class_dir.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

    if len(images) < 3:
        print(f"  ⚠ Skipping '{class_name}': only {len(images)} image(s), need at least 3.")
        continue

    # First split: train vs (val + test)   → 70% / 30%
    train_files, temp_files = train_test_split(
        images, test_size=(VAL_RATIO + TEST_RATIO), random_state=SEED, shuffle=True
    )
    # Second split: val vs test             → 15% / 15% of total
    val_files, test_files = train_test_split(
        temp_files, test_size=0.5, random_state=SEED, shuffle=True
    )

    # Copy (not move) files into their split directories
    for split_name, file_list in (("train", train_files), ("val", val_files), ("test", test_files)):
        dest_dir = SPLIT_ROOT / split_name / class_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for src in file_list:
            shutil.copy2(src, dest_dir / src.name)

    split_records.append({
        "Class Name":       class_name,
        "Original Count":   len(images),
        "Train Count":      len(train_files),
        "Validation Count": len(val_files),
        "Test Count":       len(test_files),
    })

df_split = pd.DataFrame(split_records)
print("\n✓ Splitting complete.")

Found 48 classes.



Splitting classes: 100%|██████████| 48/48 [04:44<00:00,  5.93s/ class]


✓ Splitting complete.


In [4]:
# Step 13.3 — Summary
total_classes = len(df_split)
total_images  = df_split["Original Count"].sum()
total_train   = df_split["Train Count"].sum()
total_val     = df_split["Validation Count"].sum()
total_test    = df_split["Test Count"].sum()

print("=" * 50)
print("DATASET SPLIT SUMMARY")
print("=" * 50)
print(f"Total Classes:      {total_classes}")
print(f"Total Images:       {total_images:,}")
print(f"Train Images:       {total_train:,}  ({total_train/total_images*100:.1f}%)")
print(f"Validation Images:  {total_val:,}  ({total_val/total_images*100:.1f}%)")
print(f"Test Images:        {total_test:,}  ({total_test/total_images*100:.1f}%)")
print("=" * 50)

DATASET SPLIT SUMMARY
Total Classes:      48
Total Images:       42,797
Train Images:       29,942  (70.0%)
Validation Images:  6,418  (15.0%)
Test Images:        6,437  (15.0%)


In [6]:
# Step 13.4 — Verify No Overlaps & Write Excel Report

# ── Overlap & presence verification ──────────────────────────────────────────
all_overlap_ok  = True
all_presence_ok = True

for row in split_records:
    cls  = row["Class Name"]
    sets = {}
    for split_name in ("train", "val", "test"):
        split_dir = SPLIT_ROOT / split_name / cls
        if not split_dir.exists():
            print(f"  ✗ Missing: {split_name}/{cls}")
            all_presence_ok = False
            sets[split_name] = set()
        else:
            sets[split_name] = {f.name for f in split_dir.iterdir() if f.is_file()}
    # Pairwise overlap check
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = sets[a] & sets[b]
        if overlap:
            print(f"  ✗ Overlap in '{cls}' between {a} & {b}: {len(overlap)} file(s)")
            all_overlap_ok = False

print("Overlap check:  ", "✓ No overlaps found."            if all_overlap_ok  else "✗ Overlaps detected!")
print("Presence check: ", "✓ All classes exist in train/val/test." if all_presence_ok else "✗ Some classes are missing!")

# ── Write Dataset_Split sheet to Excel ───────────────────────────────────────
SHEET_NAME = "Dataset_Split"

if EXCEL_FILE.exists():
    wb = load_workbook(EXCEL_FILE)
    if SHEET_NAME in wb.sheetnames:
        del wb[SHEET_NAME]
else:
    from openpyxl import Workbook
    wb = Workbook()
    wb.remove(wb.active)

ws = wb.create_sheet(SHEET_NAME)

# Headers
headers = ["Class Name", "Original Count", "Train Count", "Validation Count", "Test Count"]
ws.append(headers)

# Data rows
for row in df_split.itertuples(index=False, name=None):
    ws.append(list(row))

# Totals row
ws.append(["TOTAL", int(total_images), int(total_train), int(total_val), int(total_test)])

# Formatting
header_fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
header_font = Font(bold=True)
for cell in ws[1]:
    cell.font  = header_font
    cell.fill  = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
for cell in ws[ws.max_row]:   # bold the totals row
    cell.font = Font(bold=True)
for column in ws.columns:
    max_len = max(len(str(cell.value or "")) for cell in column)
    ws.column_dimensions[column[0].column_letter].width = min(max_len + 2, 50)
ws.freeze_panes = "A2"
if ws.max_row > 1:
    ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"

try:
    wb.save(EXCEL_FILE)
    print(f"\n✓ Saved '{SHEET_NAME}' sheet to: {EXCEL_FILE}")
except PermissionError:
    import time
    backup = EXCEL_FILE.parent / f"{EXCEL_FILE.stem}_split_backup_{int(time.time())}{EXCEL_FILE.suffix}"
    wb.save(backup)
    print(f"⚠ Excel file locked. Saved to: {backup}")

Overlap check:   ✓ No overlaps found.
Presence check:  ✓ All classes exist in train/val/test.

✓ Saved 'Dataset_Split' sheet to: D:\Projects\plantdisease\image_count_report.xlsx
